In [1]:
import os
import pandas as pd
import numpy as np

# fusion method 비교: CUE(CUE, softmax fusion) vs SUM(단순 합산, weight=1,1) vs
# Concat(모달리티별 MLP 없이 원본 feature를 concat 후 downstream MLP로 직접 입력).
# 세 결과 모두 동일한 5-fold 내부 CV 파이프라인에서 나온 것이며, 유일한 차이는 fusion 방식이다.
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'
cue_concat_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue_concat/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 세 가지 소스: (표시 이름, metrics.csv 루트 경로)
model_sources = {
    'SUM': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'SUM'),
    'Concat': os.path.join(cue_concat_base, '{dataset}', '{target}', 'v1', 'FP+IMG'),
    'CUE': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE')
}

model_order = ['SUM', 'Concat', 'CUE']


In [2]:
def load_metric(model_dir_template, dataset, target):
    """cue/cue_concat의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다.
    두 저장소 모두 동일한 aggregate_metrics 함수를 사용하므로 스키마가 동일하다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다.")


✅ 데이터 로드 완료: 24개의 항목이 수집되었습니다.


In [3]:
# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능이므로 순위 방향을 다르게 매긴다 (1등이 가장 좋은 성능).
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}


def build_ranked_table(df_all, dataset, target, metrics=metrics_of_interest, model_order=model_order):
    """지정한 dataset/target에 대해 Model별 metric 값 + 순위 + 평균 순위를 담은 표를 만든다.

    Returns:
        Optional[pd.DataFrame]: 컬럼이
            Model | avg_rank | rmse | rmse_rank | r2 | r2_rank | pcc | pcc_rank
            순서인 표. 성능 값(rmse/r2/pcc)은 소수점 셋째 자리, 순위(*_rank, avg_rank)는
            소수점 첫째 자리로 반올림한다. 해당 dataset/target에 데이터가 하나도 없으면 None.
    """
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target)]
    df_sub = df_sub.set_index('Model').reindex(model_order)
    df_sub = df_sub.dropna(subset=[m for m in metrics if m in df_sub.columns], how='all')
    if df_sub.empty:
        return None

    table = pd.DataFrame(index=df_sub.index)
    rank_cols = []
    for m in metrics:
        if m not in df_sub.columns:
            continue
        table[m] = df_sub[m].round(3)
        rank_col = f'{m}_rank'
        table[rank_col] = df_sub[m].rank(ascending=LOWER_IS_BETTER.get(m, True), method='min').round(1)
        rank_cols.append(rank_col)

    table['avg_rank'] = table[rank_cols].mean(axis=1).round(1)

    ordered_cols = ['avg_rank']
    for m in metrics:
        if m in table.columns:
            ordered_cols += [m, f'{m}_rank']
    table = table[ordered_cols]

    return table.reset_index()  # 'Model'을 첫 컬럼으로 복원


table_save_dir = os.path.join(cue_base, 'table', 'fusion_comparison')
os.makedirs(table_save_dir, exist_ok=True)

# DIS -> dis(dissociation), INH -> inh(inhibition)
dataset_filenames = {'DIS': 'dis.csv', 'INH': 'inh.csv'}

ranked_tables = {}
for d_name in datasets:
    dataset_tables = []
    for target in targets:
        table = build_ranked_table(df_total, d_name, target)
        if table is None:
            continue
        ranked_tables[(d_name, target)] = table

        print(f"\n=== [{d_name} / {target}] ===")
        print(table.to_string(index=False))

        # 맨 처음 컬럼에 Target을 추가한 뒤(targets 순서 = PI, Ssel, WS2, RS2) 이어 붙인다.
        table_with_target = table.copy()
        table_with_target.insert(0, 'Target', target)
        dataset_tables.append(table_with_target)

    if not dataset_tables:
        continue

    dataset_table = pd.concat(dataset_tables, ignore_index=True)
    save_path = os.path.join(table_save_dir, dataset_filenames[d_name])
    dataset_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"\n✅ 저장 완료: {save_path}")



=== [Kd / PI] ===
 Model  avg_rank  rmse  rmse_rank    r2  r2_rank   pcc  pcc_rank
   SUM       3.0 0.242        3.0 0.124      3.0 0.363       3.0
Concat       1.0 0.237        1.0 0.159      1.0 0.419       1.0
   CUE       2.0 0.242        2.0 0.125      2.0 0.366       2.0

=== [Kd / Ssel] ===
 Model  avg_rank  rmse  rmse_rank    r2  r2_rank   pcc  pcc_rank
   SUM       3.0 0.904        3.0 0.046      3.0 0.300       3.0
Concat       1.0 0.893        1.0 0.071      1.0 0.334       1.0
   CUE       2.0 0.895        2.0 0.065      2.0 0.314       2.0

=== [Kd / WS(alpha=2)] ===
 Model  avg_rank  rmse  rmse_rank    r2  r2_rank   pcc  pcc_rank
   SUM       2.0 0.283        2.0 0.211      2.0 0.487       2.0
Concat       3.0 0.293        3.0 0.154      3.0 0.459       3.0
   CUE       1.0 0.282        1.0 0.221      1.0 0.490       1.0

=== [Kd / RS(k=2)] ===
 Model  avg_rank  rmse  rmse_rank     r2  r2_rank   pcc  pcc_rank
   SUM       2.0 1.717        2.0 -0.030      2.0 0.285       